In [26]:
"""
Vorbereitung
step 1) erstmal Dateien extrahieren
step 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)
"""

'\nVorbereitung\nstep 1) erstmal Dateien extrahieren\nstep 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)\n'

In [11]:

import pandas as pd
import numpy as np


# name = 'produkt_zehn_min_sd_20000101_20080526_00222.csv' -> war erster Versuch

name = 'produkt_st_stunde_19800101_20250930_04466_FG_LBERG.csv'
name_tamb = 'produkt_tu_stunde_19510101_20241231_04466_Tamb_TT_TU.csv'

# CSV-Datei laden
df = pd.read_csv(name,sep=';')
df_Tamb = pd.read_csv(name_tamb,sep=';')
df_Tamb



,STATIONS_ID,MESS_DATUM,QN_9,TT_TU,RF_TU,eor
0,4466,1951010101,5,-10.0,87.0,eor
1,4466,1951010102,5,-9.1,88.0,eor
2,4466,1951010103,5,-8.8,89.0,eor
3,4466,1951010104,5,-8.1,89.0,eor
4,4466,1951010105,5,-7.4,89.0,eor
...,...,...,...,...,...,...
648661,4466,2024123119,3,4.1,93.0,eor
648662,4466,2024123120,3,4.5,91.0,eor
648663,4466,2024123121,3,5.0,88.0,eor
648664,4466,2024123122,3,5.3,87.0,eor


In [15]:
df_neu = df

df_neu['MESS_DATUM'] = pd.to_datetime(df['MESS_DATUM'], format='%Y%m%d%H:%M')
df_neu

df_neu_tamb = df_Tamb
df_neu_tamb['MESS_DATUM'] = pd.to_datetime(df_Tamb['MESS_DATUM'], format='%Y%m%d%H')
df_neu_tamb

,STATIONS_ID,MESS_DATUM,QN_9,TT_TU,RF_TU,eor
0,4466,1951-01-01 01:00:00,5,-10.0,87.0,eor
1,4466,1951-01-01 02:00:00,5,-9.1,88.0,eor
2,4466,1951-01-01 03:00:00,5,-8.8,89.0,eor
3,4466,1951-01-01 04:00:00,5,-8.1,89.0,eor
4,4466,1951-01-01 05:00:00,5,-7.4,89.0,eor
...,...,...,...,...,...,...
648661,4466,2024-12-31 19:00:00,3,4.1,93.0,eor
648662,4466,2024-12-31 20:00:00,3,4.5,91.0,eor
648663,4466,2024-12-31 21:00:00,3,5.0,88.0,eor
648664,4466,2024-12-31 22:00:00,3,5.3,87.0,eor


In [ ]:
#start = df_neu[df_neu['MESS_DATUM'] == '2004-01-01 00:00'].index --> ist schlecht lieber gleich mit pd.Timestemp Attribut arbeiten
#ende = df_neu[df_neu['MESS_DATUM'] == '2004-12-31 23:50'].index
#Versuche:
#df_neu['MESS_DATUM'] = df_neu[(df_neu['MESS_DATUM'] >= '2004-01-01 00:00') & (df_neu['MESS_DATUM'] < '2004-12-31 23:50')]
#df_neu = df_neu[(df_neu['MESS_DATUM'] >= start[0]) & (df_neu['MESS_DATUM'] <= ende[0])]
#df_neu['MESS_DATUM'][start:ende]

In [ ]:
# df_neu['MESS_DATUM'] = pd.to_datetime(df_neu['MESS_DATUM'], format='%Y-%m-%d %H:%M') # wird in Zelle oben gemacht

# eigentlich ist 2004 richtig, hier wird 2024 verwendet für Proof of concept
start = pd.Timestamp('2004-01-01 00:00')
ende = pd.Timestamp('2004-12-31 23:50')
# start = pd.Timestamp('2024-01-01 00:00')
# ende = pd.Timestamp('2024-12-31 23:50')


# Filtere den DataFrame im gewünschten Zeitraum
df_neu = df_neu[(df_neu['MESS_DATUM'] >= start) & (df_neu['MESS_DATUM'] <= ende)]
df_neu_tamb = df_neu_tamb[(df_neu_tamb['MESS_DATUM'] >= start) & (df_neu_tamb['MESS_DATUM'] <= ende)]


df_neu_tamb

datenreihen haben selbe länge


,STATIONS_ID,MESS_DATUM,QN_9,TT_TU,RF_TU,eor
464565,4466,2004-01-01 00:00:00,10,-4.6,97.0,eor
464566,4466,2004-01-01 01:00:00,10,-4.9,96.0,eor
464567,4466,2004-01-01 02:00:00,10,-5.3,96.0,eor
464568,4466,2004-01-01 03:00:00,10,-5.7,96.0,eor
464569,4466,2004-01-01 04:00:00,10,-5.0,98.0,eor
...,...,...,...,...,...,...
473344,4466,2004-12-31 19:00:00,10,3.6,95.0,eor
473345,4466,2004-12-31 20:00:00,10,3.6,96.0,eor
473346,4466,2004-12-31 21:00:00,10,3.5,96.0,eor
473347,4466,2004-12-31 22:00:00,10,3.0,96.0,eor


In [36]:
#Transition von J/cm^2 [DWD] zu W/m^2 [PVSyst]
# 
# 1W = 1J/s 
# 1m^2 = 100^2 cm^2
# alle Werte sind in 15 Minuten Schritten
# --> 1 J/cm^2 = 1/(60[s->min]*15[timestep]*100^2[Fläche]) W/m^2

energy_to_power_15min = 1/(60*15*100^2) # J/cm^2 to W/m^2 [15min]
energy_to_power_1h    = 1/(60^2 *100^2) # J/cm^2 to W/m^2 [1h] # später addiere ich ja werte und teile sie durch 4 --> braucht nur einen Transition Faktor


In [37]:
#df_neu = df_neu.drop(columns=['STATIONS_ID'])
#df_neu = df_neu.drop(columns=['QN'])

df_neu_relevant = df_neu[['MESS_DATUM','FD_LBERG']]
df_neu_relevant['GHI'] = df_neu_relevant['FD_LBERG']*energy_to_power_1h

df_neu_relevant['Year'] = df_neu['MESS_DATUM'].dt.year
df_neu_relevant['Month'] = df_neu['MESS_DATUM'].dt.month
df_neu_relevant['Day'] = df_neu['MESS_DATUM'].dt.day
df_neu_relevant['Hour'] = df_neu['MESS_DATUM'].dt.hour
df_neu_relevant['Minute'] = df_neu['MESS_DATUM'].dt.minute


#bisschen unschön, aber dann passt es: GHI ans Ende
# hier sollte es noch GS_10 heißen!
# df_neu_relevant['GHI'] = df_neu['GS_10'] # nochmal gegenchecken 


df_neu_relevant.drop(columns=['MESS_DATUM'], inplace=True)
#df_neu_relevant.drop(columns=['GS_10'], inplace=True)

df_neu_relevant

#   -->  Ziel Format: ;;;;;W/m2;W/m2;deg.C

C:\Users\CasparGross\AppData\Local\Temp\ipykernel_23804\1571854557.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['GHI'] = df_neu_relevant['FD_LBERG']*energy_to_power_1h
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_23804\1571854557.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['Year'] = df_neu['MESS_DATUM'].dt.year
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_23804\1571854557.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,FD_LBERG,GHI,Year,Month,Day,Hour,Minute
209664,0.0,0.0,2004,1,1,0,24
209665,0.0,0.0,2004,1,1,1,24
209666,0.0,0.0,2004,1,1,2,24
209667,0.0,0.0,2004,1,1,3,24
209668,0.0,0.0,2004,1,1,4,24
...,...,...,...,...,...,...,...
218443,0.0,0.0,2004,12,31,19,25
218444,0.0,0.0,2004,12,31,20,25
218445,0.0,0.0,2004,12,31,21,25
218446,0.0,0.0,2004,12,31,22,25


In [39]:
if len(df_neu_tamb) == len(df_neu):
    print("datenreihen haben selbe länge")    
else:
    print("datenreihen haben nicht dieselbe länge!!!!!!!!!!")


df_1h = df_neu_relevant
df_1h['Tamb'] =  df_neu_tamb['TT_TU'].values

df_1h.drop(columns=['FD_LBERG'], inplace=True)
df_1h




datenreihen haben selbe länge


C:\Users\CasparGross\AppData\Local\Temp\ipykernel_23804\639680998.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1h['Tamb'] =  df_neu_tamb['TT_TU'].values
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_23804\639680998.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1h.drop(columns=['FD_LBERG'], inplace=True)


,GHI,Year,Month,Day,Hour,Minute,Tamb
209664,0.0,2004,1,1,0,24,-4.6
209665,0.0,2004,1,1,1,24,-4.9
209666,0.0,2004,1,1,2,24,-5.3
209667,0.0,2004,1,1,3,24,-5.7
209668,0.0,2004,1,1,4,24,-5.0
...,...,...,...,...,...,...,...
218443,0.0,2004,12,31,19,25,3.6
218444,0.0,2004,12,31,20,25,3.6
218445,0.0,2004,12,31,21,25,3.5
218446,0.0,2004,12,31,22,25,3.0


In [7]:
# #trimestamp-shift:
# # Werte des DWD sind "Summe der Globalstrahlung der vorangegangnen 10 Minuten"
# # Werte in PV Syst haben Zeitstempel zu Anfang des Intervalls
# # Ziel ist also Verschieben des Zeitstempels um 10 Minuten zurück
# # außerdem sollten Werte auf 15-Minuten Basis berechnet werden.
# # da es sich bei Werten von DWD um Energiemenge handelt ist die sinnvollste Interpolation für 30 Minuten das 2. Zeitintervall jeweils zu halbieren und beiden anderen Zeitintervallen hinzu zu addieren.
# # Der hierbei entstehende Fehler ist dann, dass die für einen Wert tendentiell überschätzt werden und den anderen Wert tendentiell unterschätzt werden, aber in Summe ist auf jeden Fall die erzeugte Energie korrekt.

# def aus_10_min_mach_15(df): 
#     # hier versuche ich mal die neuen dfs mit indicees zu berechnen und nicht wie sonst immer mit einer for Schleife, weil das viel schneller und sinnvoller ist.
#     # dafür musste ich allerdings immer den Index neu setzen mit .reset_index(drop=True), da pd sonst keine Matrixoperationen durchführen konnte 
#     indicees_15 = [i for i in range(len(df)-1) if i % 3 == 0 or i % 3 == 2]  # ich suche für 30 Minuten immer Minute 0 und 20 aus, damit sie die neuen 15 Minuten Werte werden
#     df_15 = df.iloc[indicees_15].reset_index(drop=True) # jetzt df hiermit erzeugen
#     indicees_15_aufteilen = [i for i in range(len(df)-1) if i % 3 == 1]  # jetzt wird für  30 Minuten immer Minute 20 ausgesucht 
#     df_15_aufteilen = df['GS_10'].iloc[indicees_15_aufteilen].repeat(2).reset_index(drop=True)/2   # diese muss dann noch durch 2 geteilt werden, um die Hälfte dem ersten Wert und die andere Hälfte dem zweiten Wert zuzuordnen 
#     df_15['GS_10'] =  df_15['GS_10'] + df_15_aufteilen
#     df_15['GHI'] = df_15['GS_10'] * energy_to_power_15min
#     df_15.drop(columns=['GS_10'], inplace=True)
    
#     # hier Transition Faktor einbeziehen
    
#     #indicees_minutes = [i for i in range(len(df_15)-1) if i % 3 == 2] # hier wähle ich wieder jeden zweiten Wert aus, um die Minuten anzupassen
#     #hier musste ich leider die for-Schleife wählen, weil es irgendwarum nicht anders funktioniert hat...
#     for i in range(int(len(df_15)/2)):
#         j= i*2 +1
#         df_15['Minute'].iloc[j] = df_15['Minute'].iloc[j] -5

#     #df_15['Minute'].iloc[indicees_minutes] = df_15['Minute'] - minus_fünf   # Reste von früheren Versuchen. Wenn der Code mal automatisiert irgendwo eingebaut werden sollte, dann müsste man hier noch die for-Schleife erstezen.
#     #df_1h = df_15.iloc[::2].reset_index(drop=True)  # hier wähle ich dann immer jeden zweiten Wert aus, um die Stundenwerte zu bekommen
    
#     return df_15 
    
# ende

# df_15   = aus_10_min_mach_15(df_neu_relevant)
# df_15


In [8]:

# # # indicees_fuer_1h = np.zeros([3,round(len(df_15)/4)])  # nicht wirklich resillient, da indicee-Längen manchmal unterschiedlich sind.


# # hier einmal indicees erstellen, damit keine for-Schleife genutzt werden muss. --> Logik ist einfach: aus 15_min_Werten müssen 1h Werte entstehen --> wir wollen immer einstündig versetzte Einträge 
# indicees_fuer_1h_1= np.array([i for i in range(len(df_15)-1) if i % 4 == 0])
# indicees_fuer_1h_2= np.array([i for i in range(len(df_15)-1) if i % 4 == 1])
# indicees_fuer_1h_3= np.array([i for i in range(len(df_15)-1) if i % 4 == 2])
# indicees_fuer_1h_4= np.array([i for i in range(len(df_15)-1) if i % 4 == 3])


# # hier einmal den df für 1h erstellen, indem wir die Einträge der vollen Stunde übernehmen.
# df_1h = df_15.iloc[indicees_fuer_1h_1].reset_index(drop=True)  
# # dann als nächstes die Globalstrahlung der 15 Minuten Werte aufaddieren und durch 4 teilen (ist ja ne Leistung) --> hierfür haben wir ja schon die Indicees und .reset_index ist wichtig, weil pd die df nicht matchen kann 
# df_1h['GHI'] = (df_15['GHI'].iloc[indicees_fuer_1h_1].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_2].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_3].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_4].reset_index(drop=True)) /4


# # Muss unbedingt noch geändert werden, nur für Proof of concept

# df_1h['Tamb'] = 15

# df_1h
# # hier sieht man allerdings schon die Ungenauigkeit des Stundensystems: während im 15 Minuten System der Sonnenaufgang zummindest noch auf 7:30 Uhr angesetzt ist, ist er im Stundensystem schon auf 7:00 Uhr gesetzt.


In [42]:
# W/m^2 ist auf alle Fälle nicht J/cm^2 --> es muss also noch Faktor berücksichtigt werden
new_row = ['W/m2', '', '', '', '','', 'deg.C']   # 'deg.C' und gesamte Spalte für Temp fehlt noch

# einheiten einfügen bei CSV -> entkommentieren der beiden Zeilen unten
# df_15 = pd.concat([pd.DataFrame([new_row], columns=df_15.columns), df_15], ignore_index=True) # nur einmal einfügen
df_1h = pd.concat([pd.DataFrame([new_row], columns=df_1h.columns), df_1h], ignore_index=True) # nur einmal einfügen
df_1h


,GHI,Year,Month,Day,Hour,Minute,Tamb
0,W/m2,,,,,,deg.C
1,,,,,,W/m2,deg.C
2,0.0,2004,1,1,0,24,-4.6
3,0.0,2004,1,1,1,24,-4.9
4,0.0,2004,1,1,2,24,-5.3
...,...,...,...,...,...,...,...
8781,0.0,2004,12,31,19,25,3.6
8782,0.0,2004,12,31,20,25,3.6
8783,0.0,2004,12,31,21,25,3.5
8784,0.0,2004,12,31,22,25,3.0


In [43]:
#achtung! name muss noch im cache sein!

name_new = name.replace('.csv', '_2004_1h_for_pv_syst.csv')

# nur ausführen wenn Datei ready ist, sonst wird bestehende Datei überschrieben 
df_1h.to_csv(name_new, sep=';', index=False, header=['GHI','Year','Month','Day','Hour','Minute','Tamb'])



#   diese Header am besten noch integrieren, geht aber auch händisch

#Site; Schleswig
#Country; germany
#Data Source; DWD_station_4466 
#Time step; 1h
#Year; 2024
#Latitude; 54.5275
#Longitude; 9.5487
#Altitude; 42.70
#Time Zone; 1  

# --> muss aber individuell je nach Metadaten angepasst werden (könnte man aber in Zukunft automatisieren)

# Zielformat
# Year;Month;Day;Hour;Minute;GHI;Tamb;
# ;;;;;W/m2;deg.C;

In [ ]:
"""
ToDo:

# location checken
# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.

# Simulieren mit Lima

# in Zukunft Interpolieren, wenn Werte fehlen!!! -> hat jetzt durch Zufall auch ohne funktioniert, aber kann man sich nicht 100%ig drauf verlassen

# Wetterstationen filtern später

UTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert
wenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht

https://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data 
If there is a time shift varying along the year:

With the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").
Otherwise there is no practical way, except modifying the dates directly in the source file.
If the file is written in UT (Universal Time):

With the general ASCII importing tool, check the box "Universal time".
With PVsyst standard format, you can add a line #Time Reference with the value UT

"""

'\nToDo:\n\n# location checken\n# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.\n\n# Simulieren mit Lima\n\n# Wetterstationen filtern später\n\nUTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert\nwenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht\n\nhttps://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data \nIf there is a time shift varying along the year:\n\nWith the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").\nOtherwise there is no practical way, except modifying the dates directly in the source file.\nIf the file is written in UT (Universal Time):\n\nWith the general ASCII importing tool, check the 